# ErrorConsumer 테스트

error_data topic에서 에러 메시지 읽어 PostgreSQL에 저장

In [ ]:
import sys
sys.path.insert(0, '../..')

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

from src.kafka.error_consumer import ErrorDataConsumer
from config.settings import KAFKA_BROKERS, POSTGRES_HOST, POSTGRES_PORT, POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD

print("📌 설정:")
print(f"  Kafka: {','.join(KAFKA_BROKERS)}")
print(f"  PostgreSQL: {POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

## 1. ErrorConsumer 초기화

In [ ]:
try:
    postgres_config = {
        'host': POSTGRES_HOST,
        'port': POSTGRES_PORT,
        'database': POSTGRES_DB,
        'user': POSTGRES_USER,
        'password': POSTGRES_PASSWORD
    }
    
    consumer = ErrorDataConsumer(
        bootstrap_servers=','.join(KAFKA_BROKERS),
        group_id='error_consumer_test',
        postgres_config=postgres_config
    )
    
    print("✅ ErrorConsumer 초기화 성공!")
except Exception as e:
    print(f"❌ 초기화 실패: {e}")

## 2. error_data topic 읽기 (최대 10개)

In [ ]:
try:
    errors = consumer.consume_errors(max_records=10, timeout_ms=5000)
    
    print(f"✅ {len(errors)}개의 에러 메시지 수신")
    print()
    
    if errors:
        print("샘플 에러 메시지:")
        for i, error in enumerate(errors[:3], 1):
            print()
            print(f"  {i}. 에러 타입: {error['error_type']}")
            print(f"     시간: {error['timestamp']}")
            print(f"     상태: {error['status']}")
            print(f"     메시지: {error['message'][:50]}...")
    else:
        print("⚠️  처리할 에러가 없습니다 (정상)")
except Exception as e:
    print(f"❌ 읽기 실패: {e}")

## 3. 에러를 PostgreSQL에 저장

In [ ]:
try:
    if errors:
        stored_count = consumer.store_errors_to_db(errors)
        print(f"✅ {stored_count}/{len(errors)}개 에러 저장 완료")
    else:
        print("ℹ️  저장할 에러가 없습니다")
except Exception as e:
    print(f"❌ 저장 실패: {e}")

## 4. 통계 계산

In [ ]:
try:
    stats = consumer.calculate_statistics()
    
    print("📊 에러 통계 (지난 1시간):")
    print(f"  총 에러: {stats['total_errors']}개")
    print(f"  Critical: {stats['critical_count']}개")
    print(f"  Pending: {stats['pending_count']}개")
    
    if stats['by_error_type']:
        print()
        print("  에러 타입별:")
        for error_type, count in sorted(stats['by_error_type'].items(), key=lambda x: x[1], reverse=True):
            print(f"    - {error_type}: {count}개")
    
    if stats['by_module']:
        print()
        print("  모듈별:")
        for module, count in sorted(stats['by_module'].items(), key=lambda x: x[1], reverse=True):
            print(f"    - {module}: {count}개")
except Exception as e:
    print(f"❌ 통계 계산 실패: {e}")

## 5. 통계를 PostgreSQL에 저장

In [ ]:
try:
    if stats['total_errors'] > 0:
        stored_count = consumer.store_statistics_to_db(stats)
        print(f"✅ {stored_count}개 통계 저장 완료")
    else:
        print("ℹ️  저장할 통계가 없습니다 (에러 없음)")
except Exception as e:
    print(f"❌ 통계 저장 실패: {e}")

## 6. error_logs 테이블 확인

In [ ]:
import pandas as pd
import psycopg2

try:
    conn = psycopg2.connect(**postgres_config)
    
    df = pd.read_sql_query(
        "SELECT * FROM error_logs ORDER BY created_at DESC LIMIT 5",
        conn
    )
    
    print("✅ error_logs 최근 5개:")
    print()
    print(df[['timestamp', 'error_type', 'module', 'status', 'created_at']].to_string())
    
    conn.close()
except Exception as e:
    print(f"❌ 조회 실패: {e}")

## 7. 정리

In [ ]:
try:
    consumer.close()
    print("✅ Consumer 종료 완료")
except Exception as e:
    print(f"❌ 종료 실패: {e}")